In [2]:
# Shared reasoning-analysis pipeline for Appendix Tables A2-A7.
# This cell uses the same extractor, tokenizer, lexicons, and vocabulary
# matcher as Table 5. It writes derived outputs only when the cell is run.

from pathlib import Path
import sys

def find_reasoning_dir():
    candidates = [
        Path.cwd(),
        Path.cwd() / "Analyses" / "Reasoning Analysis",
        Path.cwd().parent / "Reasoning Analysis",
    ]
    for candidate in candidates:
        if (candidate / "reasoning_analysis_core.py").exists():
            return candidate.resolve()
    raise FileNotFoundError("Could not locate reasoning_analysis_core.py")

REASONING_DIR = find_reasoning_dir()
if str(REASONING_DIR) not in sys.path:
    sys.path.insert(0, str(REASONING_DIR))

from reasoning_analysis_core import load_model_rationales, summarize_by_model_option

csv_files = [
    "aya-expanse-32b_test.csv",
    "gemma3-12b_test.csv",
    "gemma3-27b_test.csv",
    "gpt4o_test.csv",
    "llama3.1_test.csv",
    "llama3.2-latest_test.csv",
    "llama3.3-latest_test.csv",
    "mistral-small-3.2-24b_test.csv",
    "mistral-small3.1-latest_test.csv",
    "qwen2.5-14b_test.csv",
    "qwen2.5-32b_test.csv",
    "qwen2.5-72b_test.csv",
]

OUT_DIR = REASONING_DIR / "out"
COMPARISON_DIR = REASONING_DIR.parent / "comparisons"
OUT_DIR.mkdir(exist_ok=True)
COMPARISON_DIR.mkdir(exist_ok=True)

extracted = load_model_rationales(csv_files, data_dir=REASONING_DIR)
extracted_path = OUT_DIR / "extracted_model_option_reasonings.csv"
extracted.to_csv(extracted_path, index=False)
print(f"[OK] wrote {extracted_path} ({len(extracted)} rows)")

summary = summarize_by_model_option(extracted)
summary_path = OUT_DIR / "metrics_by_model_option.csv"
summary.to_csv(summary_path, index=False)
print(f"[OK] wrote {summary_path}")

for opt in ["A", "B", "C"]:
    sub = summary[summary["option"] == opt].copy()
    comp_path = COMPARISON_DIR / f"option_{opt}_comparison.csv"
    sub.to_csv(comp_path, index=False)
    print(f"[OK] wrote {comp_path}")

display_cols = [
    "model", "option", "n_reasonings", "avg_words", "lexical_diversity",
    "hedge_rate_per100w", "confidence_rate_per100w",
    "pct_with_justification_marker", "moral_terms_per1000w",
    "relational_terms_per1000w", "epistemic_terms_per1000w",
    "avg_sentiment_lexicon",
]
summary[display_cols].head(12)


[OK] wrote out\extracted_model_option_reasonings.csv (1437 rows)
[OK] wrote out\metrics_by_model_option.csv
[OK] wrote out\comparisons\option_A_comparison.csv
[OK] wrote out\comparisons\option_B_comparison.csv
[OK] wrote out\comparisons\option_C_comparison.csv


,model,option,n_reasonings,avg_words,lexical_diversity,hedge_rate_per100w,confidence_rate_per100w,pct_with_justification_marker,moral_terms_per1000w,relational_terms_per1000w,epistemic_terms_per1000w,avg_sentiment_lexicon
0,aya-expanse-32b,A,21,28.048,0.8693,0.679,1.358,9.52,13.582,22.071,8.489,0.429
1,gemma3-12b,A,11,17.091,0.9574,1.064,0.000,0.00,37.234,26.596,37.234,0.091
2,gemma3-27b,A,14,17.929,0.9482,1.594,0.000,0.00,11.952,23.904,31.873,0.000
3,gpt4o,A,13,15.538,0.9604,0.000,0.000,0.00,19.802,24.752,24.752,0.077
4,llama3.1,A,15,15.933,0.9833,1.674,0.418,0.00,29.289,25.105,33.473,0.133
5,llama3.2-latest,A,10,18.500,0.9459,1.622,1.081,0.00,37.838,27.027,32.432,0.300
6,llama3.3-latest,A,14,19.214,0.9257,1.115,0.743,0.00,37.175,37.175,14.870,0.071
7,mistral-small-3.2-24b,A,16,13.000,0.9471,1.442,0.000,6.25,28.846,52.885,43.269,-0.062
8,mistral-small3.1-latest,A,14,11.786,0.9697,2.424,0.606,0.00,48.485,36.364,48.485,0.071
9,qwen2.5-14b,A,7,10.857,1.0000,0.000,0.000,0.00,52.632,13.158,39.474,0.143


In [5]:
# Make Appendix-table artifacts from the shared reasoning-analysis outputs.
# Plot writing is disabled by default to avoid touching figure artifacts while
# regenerating Appendix Tables A2-A7.

from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt

def find_reasoning_dir():
    candidates = [
        Path.cwd(),
        Path.cwd() / "Analyses" / "Reasoning Analysis",
        Path.cwd().parent / "Reasoning Analysis",
    ]
    for candidate in candidates:
        if (candidate / "reasoning_analysis_core.py").exists():
            return candidate.resolve()
    raise FileNotFoundError("Could not locate reasoning_analysis_core.py")

REASONING_DIR = find_reasoning_dir()
if str(REASONING_DIR) not in sys.path:
    sys.path.insert(0, str(REASONING_DIR))

COMPARISON_DIR = REASONING_DIR.parent / "comparisons"
TABLE_DIR = REASONING_DIR / "overleaf_tables"
FIG_DIR = REASONING_DIR / "overleaf_figs"
WRITE_PLOTS = False

TABLE_DIR.mkdir(exist_ok=True)
if WRITE_PLOTS:
    FIG_DIR.mkdir(exist_ok=True)

INFILES = {
    "A": COMPARISON_DIR / "option_A_comparison.csv",
    "B": COMPARISON_DIR / "option_B_comparison.csv",
    "C": COMPARISON_DIR / "option_C_comparison.csv",
}

TABLE_COLS = [
    "model",
    "n_reasonings",
    "avg_words",
    "lexical_diversity",
    "hedge_rate_per100w",
    "confidence_rate_per100w",
    "pct_with_justification_marker",
    "moral_terms_per1000w",
    "relational_terms_per1000w",
    "epistemic_terms_per1000w",
    "avg_sentiment_lexicon",
]
PLOT_METRICS = [
    "avg_words",
    "hedge_rate_per100w",
    "confidence_rate_per100w",
    "pct_with_justification_marker",
    "avg_sentiment_lexicon",
]


def load_option_df(path, option_label):
    df = pd.read_csv(path)
    if "option" not in df.columns:
        df["option"] = option_label
    keep = [c for c in TABLE_COLS if c in df.columns]
    return df[["model", "option"] + [c for c in keep if c not in ("model", "option")]]


def save_latex_table(df, option_label):
    cols = [c for c in TABLE_COLS if c in df.columns]
    df = df[["model"] + [c for c in cols if c != "model"]].sort_values("model")
    out_path = TABLE_DIR / f"option_{option_label}_summary.tex"
    df.to_latex(out_path, index=False, float_format="%.3f", escape=True)
    print(f"[OK] LaTeX table: {out_path}")


def save_bar_charts(df, option_label):
    if not WRITE_PLOTS:
        return
    for metric in PLOT_METRICS:
        if metric not in df.columns:
            continue
        sub = df[["model", metric]].dropna().sort_values(metric, ascending=False)
        plt.figure()
        plt.bar(sub["model"], sub[metric])
        plt.xticks(rotation=45, ha="right")
        plt.ylabel(metric)
        plt.title(f"{metric} - Option {option_label}")
        plt.tight_layout()
        out_path = FIG_DIR / f"{metric}_option_{option_label}.png"
        plt.savefig(out_path, dpi=200)
        plt.close()
        print(f"[OK] Figure: {out_path}")


all_frames = []
for opt, path in INFILES.items():
    if not path.exists():
        print(f"[WARN] Missing: {path}")
        continue
    df = load_option_df(path, opt)
    all_frames.append(df)
    save_latex_table(df, opt)
    save_bar_charts(df, opt)

if all_frames:
    all_df = pd.concat(all_frames, ignore_index=True)
    combined_csv = TABLE_DIR / "all_options_summary.csv"
    all_df.to_csv(combined_csv, index=False)
    print(f"[OK] Combined CSV: {combined_csv}")


[OK] LaTeX table: overleaf_tables\option_A_summary.tex
[OK] Figure: overleaf_figs\avg_words_option_A.png
[OK] Figure: overleaf_figs\hedge_rate_per100w_option_A.png
[OK] Figure: overleaf_figs\confidence_rate_per100w_option_A.png
[OK] Figure: overleaf_figs\pct_with_justification_marker_option_A.png
[OK] Figure: overleaf_figs\avg_sentiment_lexicon_option_A.png
[OK] LaTeX table: overleaf_tables\option_B_summary.tex
[OK] Figure: overleaf_figs\avg_words_option_B.png
[OK] Figure: overleaf_figs\hedge_rate_per100w_option_B.png
[OK] Figure: overleaf_figs\confidence_rate_per100w_option_B.png
[OK] Figure: overleaf_figs\pct_with_justification_marker_option_B.png
[OK] Figure: overleaf_figs\avg_sentiment_lexicon_option_B.png
[OK] LaTeX table: overleaf_tables\option_C_summary.tex
[OK] Figure: overleaf_figs\avg_words_option_C.png
[OK] Figure: overleaf_figs\hedge_rate_per100w_option_C.png
[OK] Figure: overleaf_figs\confidence_rate_per100w_option_C.png
[OK] Figure: overleaf_figs\pct_with_justification_ma